In [1]:
# https://judge.nitro-ai.org/competitions/rm-olimpiada-ai/simulation-1/1/view

import os 
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
from tqdm.auto import tqdm 

/Users/Abu/coding-workspace/Coding/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pd.read_csv("train_data.csv")
test = pd.read_csv("test_data.csv")
subm = pd.read_csv("sample_output.csv")

train.shape, test.shape, subm.shape 

((159571, 8), (153164, 2), (153177, 3))

In [3]:
train

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,1,And about you seeing somebody dressed up as Bi...,0,0,0,0,0,0
1,2,Woo-hoo2 \n\nAnd now the Jonathan Wild is seri...,0,0,0,0,0,0
2,3,This is directe at the AC:\n *Wikipedia talk:R...,0,0,0,0,0,0
3,4,Have you all gone insane? Reggie Jackson is no...,0,0,0,0,0,0
4,5,"""\nPlease stop adding nonsense to Wikipedia. I...",0,0,0,0,0,0
...,...,...,...,...,...,...,...,...
159566,159567,Really distastefull opinonated personnal attac...,0,0,0,0,0,0
159567,159568,"""\n""""Anal raping of donkeys""""?? It must be a j...",1,0,1,0,0,0
159568,159569,Rename at eswikibooks \n\nDone. Left some note...,0,0,0,0,0,0
159569,159570,"""\n\n Ping \n""""ping me when the case is finish...",0,0,0,0,0,0


In [4]:
feature = 'comment_text'

targets = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

In [5]:
subm.iloc[:6, 2] = train[targets].sum()

In [6]:
subm.iloc[6:13, 2] = train[targets].sum(axis=1).value_counts().sort_index()

In [7]:
from sklearn.model_selection import train_test_split 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def solve(target):
    train_indices, valid_indices = train_test_split(np.arange(len(train)), stratify=train[target], test_size=0.1, random_state=42)
    vectorizer = TfidfVectorizer(max_features=None, ngram_range=(1, 1))
    X_train = vectorizer.fit_transform(train.loc[train_indices, feature])
    X_valid = vectorizer.transform(train.loc[valid_indices, feature])
    X_test = vectorizer.transform(test[feature])
    y_train, y_valid = train[target][train_indices], train[target][valid_indices]

    model = LogisticRegression().fit(X_train, y_train)
    score = roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])
    test_pred = model.predict_proba(X_test)[:, 1]
    return test_pred, score

In [8]:
preds = np.zeros((len(test), 6))
scores = []

for i, target in enumerate(targets):
    preds[:, i], score = solve(target)
    scores.append(score)
    print(f'Target {target} score: {score:.4f}')

print(f'Average score: {sum(scores)/len(scores):.4f}')

Target toxic score: 0.9741
Target severe_toxic score: 0.9861
Target obscene score: 0.9782
Target threat score: 0.9828
Target insult score: 0.9729
Target identity_hate score: 0.9787
Average score: 0.9788


In [9]:
subm.iloc[13:, 2] = [' '.join(str(p) for p in pred)for pred in preds]

In [10]:
subm.to_csv("submission.csv", index=False)
subm

,subtaskID,datapointID,answer
0,1,toxic,15294
1,1,severe_toxic,1595
2,1,obscene,8449
3,1,threat,478
4,1,insult,7877
...,...,...,...
153172,3,312731,0.9999998067993854 0.8512715459675034 0.999999...
153173,3,312732,0.011292099105520566 0.004221988839162394 0.01...
153174,3,312733,0.02391171861587366 0.005009724101376174 0.015...
153175,3,312734,0.03355530365508968 0.002327969389956016 0.020...
